# 4 — Robustez, rich-club e o efeito da agregação

Este notebook substitui as análises que **perderam sentido** com a mudança de unidade:

| Saiu | Por quê | Entrou no lugar |
|---|---|---|
| Lei de potência (α) | 145 nós não sustentam ajuste de cauda | desigualdade de volume (Gini/Lorenz, NB2) |
| Small-world (σ) | rede densa: caminho médio já é ~1,4 | — |
| k-core | trivializa em rede densa | s-core por força (NB2) e rich-club aqui |
| Componente gigante | tudo é uma componente só | — |
| Robustez por fragmentação | a rede nunca fragmenta | perda de **eficiência ponderada** |

## Preparação

In [ ]:
import sys
from pathlib import Path

# permite rodar o notebook a partir de notebooks/ usando os módulos de src/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.utils import load_config
from src.exporter import InlineExporter
from src import antenna

CIDADE = "campinas"   # troque aqui: precisa existir config/<cidade>.yaml

config = load_config(CIDADE)
config["spatial"]["download_basemap"] = True   # False para rodar offline

edges_antenna = pd.read_parquet(ROOT / config["data"]["edges_antenna_path"])
antennas = pd.read_parquet(ROOT / config["data"]["antennas_path"])

net = antenna.build(edges_antenna, antennas, config)
ex = InlineExporter(config)
print(f"{net.n_antennas} regiões | {net.G.number_of_edges()} fluxos | "
      f"{net.nodes['n_users'].sum():,} moradores agregados")

## Robustez e rich-club

A rede de regiões não se parte quando perde uma área — ela perde capacidade aos poucos. Por
isso a robustez é medida pela queda da eficiência de comunicação ponderada, não pelo tamanho
da componente gigante.

In [ ]:
from src.pipeline import advanced

resultado = advanced.run(net, edges_antenna, config, ex)

## O ponto mais importante: falácia ecológica

O achado antigo do projeto era: *"49% das chamadas ligam pessoas do mesmo quintil, contra 26%
esperado ao acaso — 1,9×"*.

Esse modelo nulo embaralha o quintil **entre pessoas**, e ao fazer isso destrói também o fato
de que vizinhos compartilham o quintil simplesmente por morarem no mesmo lugar. Refazendo a
conta com um nulo que embaralha o quintil **entre regiões** — preservando quem mora com quem —
a razão praticamente desaparece.

A conclusão muda de figura: a "segregação socioeconômica na comunicação" é, em quase toda a
sua extensão, **segregação territorial**.

In [ ]:
m = resultado["metrics"]
pd.DataFrame({
    "fração do volume no mesmo quintil": [
        m["individual_homophily_observed"],
        m["individual_homophily_null_by_user"],
        m["individual_homophily_null_by_region"],
    ]
}, index=["observado", "acaso (quintil entre pessoas)", "acaso (quintil entre regiões)"]).round(3)

In [ ]:
print(f"razão com nulo ingênuo:    {m['individual_homophily_ratio_naive']:.2f}x")
print(f"razão com nulo territorial: {m['individual_homophily_ratio_spatial_null']:.2f}x")

### Por que isso é bom para a apresentação

Não é um achado negativo — é um achado **mais forte e mais acionável**. Ele diz que a
desigualdade da comunicação tem endereço: a política que integra estratos sociais não é sobre
indivíduos, é sobre **conectar territórios**.